# SUPPORT2 Dataset: Bayesian Modeling
**AAI-500 Applied Statistics for AI | Final Team Project**

---

## Notebook Objectives

This notebook covers the **Model Selection & Analysis** phase using a **Bayesian Logistic Regression**
implemented in PyMC. For some lab features that are nonlinear, we explore modeling them using B-Spline. 

**Why Bayesian Logistic Regression?**
- LR is simpliest in explainability and quick to run. Most of our features behaves linearly. Some exception are a few lab values which we will address using nonlinear models.
- Unlike regular LR whose coefficients are point estimates, BLR Produces a posterior distribution over each coefficient.
- Provides calibrated uncertainty intervals for every prediction, a necessity for clinical domains.


---
## Section 0: Setup & Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns
from patsy import build_design_matrices, dmatrix
from scipy.special import expit as sigmoid
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

_here = Path.cwd()
_proj_root = _here if (_here / "utils").exists() else (_here / "../..").resolve()
if str(_proj_root) not in sys.path:
    sys.path.insert(0, str(_proj_root))

from utils.dataset import load_csv  # noqa: E402
from utils.evaluation import (  # noqa: E402
    auc_table,
    brier_score,
    get_benchmark_preds,
    plot_roc,
)

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

---
## Section 1: Load Data & Feature Selection

We use **12 continuous, 1 binary, and 3 categorical** features (25 model columns after
one-hot encoding), based on EDA findings and the feature-selection experiments in Section 1b below:

| Feature | Type | EDA justification |
|---|---|---|
| `age` | continuous | Non-survivors were ~3 years older on average |
| `meanbp` | continuous | Hemodynamic instability - direct mortality signal |
| `bili` | continuous | Liver/sepsis marker; heavy right-skew with a long high-value tail (high IQR-outlier rate in EDA Section 2), linked with liver failure |
| `bun` | continuous | Kidney function marker |
| `alb` | continuous | Nutritional/liver status; low in high-mortality groups |
| `adlsc` | continuous | Functional status - non-survivors more dependent (2.27 vs 1.55) |
| `scoma` | continuous | SUPPORT coma score - severity of neurological impairment |
| `pafi` | continuous | PaO2/FiO2 ratio (oxygenation); added in Section 1b AUC search |
| `hrt` | continuous | Heart rate; added in Section 1b AUC search |
| `dzgroup` | categorical | Knaus et al combined dzgroup into dzclass. We first tried dzclass but dzgroup gave us moderate increase in AUC so we deviate from Knaus here.
| `ca` | categorical | Cancer status (no/yes/metastatic); added in Section 1b AUC search |
| `resp` | continuous | Respiratory rate (day-3 vital sign) |
| `temp` | continuous | Body temperature (day-3 vital sign) |
| `sod` | continuous | Serum sodium (electrolyte balance) |
| `diabetes` | binary | Diabetes comorbidity flag (0/1) |
| `income` | categorical | Income bracket (socioeconomic status) |

---
## Section 1b: Feature Selection Experimentation

We use 25 model features (12 continuous + 1 binary + 3 one-hot-encoded categoricals). 

**TODO**: explain feature selection experiments

### Handling categorical features
In order for regression models to work on categorical features, we need to one-hot encode them.
This prevents strict multicollinearity (dummy variable trap)

- **dzgroup**: We drop the first disease group `ARF/MOSF w/Sepsis` and use it as the baseline reference
- **ca**: We drop metastatic and use it as the baseline cancer status

In [ ]:
df = load_csv("support2_cleaned.csv")
print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns")

TARGET_COL = "death_180d"
TRAIN_FEATURES = [
    "age",
    "meanbp",
    "bili",
    "bun",
    "alb",
    "adlsc",
    "scoma",
    "pafi",
    "hrt",
    "resp",
    "temp",
    "sod",
]

# Binary comorbidity flags (already 0/1; kept unstandardized like the dummies)
BINARY_FEATURES = ["diabetes"]

# reorganize so we can use cancer=no as baseline which makes the forest plot more sensible
df["ca"] = pd.Categorical(df["ca"], categories=["no", "yes", "metastatic"])

## Handling Categorical classes

# One-hot encodings
dzgroup_dummies = pd.get_dummies(df["dzgroup"], prefix="dzgroup", drop_first=True)
ca_dummies = pd.get_dummies(df["ca"], prefix="ca", drop_first=True)
income_dummies = pd.get_dummies(df["income"], prefix="income", drop_first=True)

DUMMY_COLS = (
    BINARY_FEATURES
    + list(dzgroup_dummies.columns)
    + list(ca_dummies.columns)
    + list(income_dummies.columns)
)
FEATURE_NAMES = TRAIN_FEATURES + DUMMY_COLS

print("dzgroup reference level: ARF/MOSF w/Sepsis")
print("ca reference level: ca_no")
print("income reference level: $11-$25k")
print(f"Dummy/binary columns: {DUMMY_COLS}")
print(f"Total features: {len(FEATURE_NAMES)}")

df_model = pd.concat(
    [
        df[TRAIN_FEATURES + BINARY_FEATURES + [TARGET_COL]],
        dzgroup_dummies,
        ca_dummies,
        income_dummies,
    ],
    axis=1,
)
print("Class balance:")
print(df_model[TARGET_COL].value_counts().rename({0: "Survived", 1: "Died"}))

---
## Section 2: Preprocessing

### Standardizing
To make the continuous features more comparable we employ standardization.
Here we rescale our continuous features coefficients using **z-score standardized**: 

$$z = \frac{x - \bar{x}}{s}$$

This centers each feature at mean 0 and rescale by a standard deviation of 1.

### Data Splitting
Then we do a 80/20 stratified split on the dataset, where 80% is used for training and 20% for testing. We picked 80/20 instead of the better 3-way split with K-fold cross-validation as it's simpler and apt for this small project.


In [ ]:
X_continuous = df_model[TRAIN_FEATURES].values
X_categorical = df_model[DUMMY_COLS].values.astype(float)
y = df_model[TARGET_COL].values

# Standardize continuous features only
scaler = StandardScaler()
X_standardized = np.column_stack([scaler.fit_transform(X_continuous), X_categorical])
n_features = X_standardized.shape[1]

# Stratified 80/20 split
idx_all = np.arange(len(df_model))
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_standardized, y, idx_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Train: {len(y_train):,}  |  Test: {len(y_test):,}")
print(
    f"Train prevalence: {y_train.mean():.1%}  |  Test prevalence: {y_test.mean():.1%}"
)

# Descriptive stats of standardized features
feat_stats = pd.DataFrame(X_train, columns=FEATURE_NAMES).describe().T
feat_stats[["mean", "std", "min", "max"]]

---
## Section 3: Prior Predictive Check

A prior represent our assumptions about the model's parameters before training.
A good prior should produce predicted probabilities spread across [0, 1]
rather than collapsing near 0 or 1.

Here we do a quick check to see if our priors are reasonable. 

**Histogram shows a nice spread confirming that our features are nicely calibrated **

**Prior specification:**
- Intercept: `Normal(0, 2)`: gives a wide range, with baseline mortality probabilities between 2% ~ 98%
- Coefficients: `Normal(0, 1)`:  penalize one feature from making a large impact. We assume in biology it's implausible for one lab metric to cause death.

In [ ]:
with pm.Model() as blr:
    X_data = pm.Data("X_data", X_train)

    # Priors
    intercept = pm.Normal("intercept", mu=0, sigma=2)
    betas = pm.Normal("betas", mu=0, sigma=1, shape=n_features)

    # Linear predictor + sigmoid link
    logit_p = intercept + pm.math.dot(X_data, betas)
    p = pm.Deterministic("p", pm.math.invlogit(logit_p))

    # Likelihood
    obs = pm.Bernoulli("obs", p=p, observed=y_train)

    prior_pred = pm.sample_prior_predictive(draws=200, random_seed=RANDOM_SEED)

prior_p = prior_pred.prior["p"].values.reshape(-1, len(y_train))
sample_probs = prior_p[:500, 0]  # distribution for one patient

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample_probs, bins=40, edgecolor="white", density=True)
ax.set_title(
    "Prior Predictive: Distribution of Predicted Probability (single patient)",
    fontweight="bold",
)
ax.set_xlabel("Predicted P(death within 180d)")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
pm.model_to_graphviz(blr)

---
## Section 4: Model Training (MCMC - NUTS Sampler)

Sampling is special to Bayesian as it's done to quantify uncertainty. 
Unlike gradient descent which finds a single best sets of weights, Bayesian samples a thousand sets of weights.

We use the **No-U-Turn Sampler (NUTS)**, a popular gradient-based MCMC (Markov Chain Monte Carlo) algorithm that
efficiently explores high-dimensional posterior distributions. It's the default sampler for PyMC.

**Sampling parameters:**

**TODO**: tweak and optimize

- `draws=1000`: posterior samples per chain
- `tune=1000`: warm-up/adaptation steps (discarded)
- `chains=4`: independent chains for convergence diagnostics
- `target_accept=0.9`: acceptance rate target (reduces divergences)

In [ ]:
with blr:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )
print("Sampling complete.")

---
## Section 5: Sanity Check on NUTS Sampler

**Key convergence diagnostics:**

| Metric | Good threshold | Meaning | Our Results
|---|---|---|--|
| **R-hat** | < 1.01 | Chains converged to the same distribution | Good
| **ESS bulk** | > 400 | Effective sample size - independent posterior draws | Good
| **Divergences** | 0 | physics error | Good


In [ ]:
summary = az.summary(trace, var_names=["intercept", "betas"])
summary.index = ["intercept"] + FEATURE_NAMES
diag_cols = [c for c in ["mean", "sd", "r_hat", "ess_bulk"] if c in summary.columns]
print(summary[diag_cols].to_string())
print(f"\nDivergences: {trace.sample_stats.diverging.values.sum()}")

---
## Section 6: Posterior Coefficient Analysis

The **94% Highest Density Interval (HDI)** is the Bayesian analogue of a confidence interval.
In the forest plot below:
* dot: mean of posterior distribution
* line: 94% Highest Density Interval. Effectively saying "We are confident 94% of true weights falls within this line"

**Reading the forest plot**
- If the HDI **excludes 0**: the feature has a credible effect on mortality
- Positive coefficient: higher value → higher mortality probability
- Negative coefficient: higher value → lower mortality probability

Coefficients are on **standardized** continuous features: a coefficient of 1.0 means
a 1-SD increase changes the log-odds by 1.0 (≈ 2.7× odds multiplier).

**Results Interpretation**
- `scoma`: strongest risk factor
- `ca_yes`, `ca_metastatic`: higher risk relative to the `ca_no` (no-cancer) baseline
- `dzgroup_*` coefficients are relative to the `ARF/MOSF w/Sepsis` baseline group
- `dzgroup_Coma`, `bun`, `alb`: less certain (94% interval near 0)

In [ ]:
# Manual forest plot using posterior arrays
post_int_flat = trace.posterior["intercept"].values.flatten()
post_b_flat = trace.posterior["betas"].values.reshape(-1, n_features)

all_names = ["intercept"] + FEATURE_NAMES
means_all = [post_int_flat.mean()] + list(post_b_flat.mean(axis=0))
lower_all = [np.percentile(post_int_flat, 3)] + list(
    np.percentile(post_b_flat, 3, axis=0)
)
upper_all = [np.percentile(post_int_flat, 97)] + list(
    np.percentile(post_b_flat, 97, axis=0)
)

fig, ax = plt.subplots(figsize=(9, 6))
y_pos = range(len(all_names))
ax.errorbar(
    means_all,
    y_pos,
    xerr=[
        np.array(means_all) - np.array(lower_all),
        np.array(upper_all) - np.array(means_all),
    ],
    fmt="o",
    color="steelblue",
    ecolor="steelblue",
    capsize=4,
    markersize=6,
    elinewidth=1.5,
)
ax.axvline(0, color="red", linestyle="--", linewidth=1.2, alpha=0.7)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(all_names)
ax.set_title(
    "Posterior Coefficient Estimates (94% Credible Interval)", fontweight="bold"
)
ax.set_xlabel("Log-Odds Coefficient")
plt.tight_layout()
plt.show()

In [ ]:
# Odds ratios from posterior means
post_betas = trace.posterior["betas"].values.reshape(-1, n_features)
or_mean = np.exp(post_betas.mean(axis=0))
or_lower = np.exp(np.percentile(post_betas, 3, axis=0))
or_upper = np.exp(np.percentile(post_betas, 97, axis=0))

or_df = pd.DataFrame(
    {
        "Feature": FEATURE_NAMES,
        "OR Mean": or_mean.round(3),
        "OR 5.5% ETI": or_lower.round(3),
        "OR 94.5% ETI": or_upper.round(3),
        "Direction": ["Higher risk" if v > 1 else "Lower risk" for v in or_mean],
    }
).sort_values("OR Mean", ascending=False)

print("Odds Ratios (per 1 SD change for continuous features):")
print(or_df.to_string(index=False))

**Note on the cancer disease groups:** the cancer-related `dzgroup` dummies (`Colon Cancer`, `Lung Cancer`, `MOSF w/Malig`) are collinear with `ca_metastatic` (those patients are all metastatic), so their coefficients are **residual** effects *given* metastatic status — read them together with `ca_metastatic`, not in isolation.

---
## Section 7: Posterior Predictive Check

Now that the model has learned from the data, we ask: *can it reproduce what we observed?* This is a **Posterior Predictive Check (PPC)**. There are two checks we can  run, from coarse to refined:

1. **Marginal rate**: does the simulated overall mortality rate match the observed rate? This is necessary but weak: per-patient errors can cancel out in the average.
2. **Calibration (reliability diagram)**: among patients the model assigns ~*x*% risk, do ~*x*% actually die? 

For brevity we're doing a simple PPC (1).

**Result**: the simulated overall rate (~46.8%) matches the observed died:survived ratio (4265:4840).

In [ ]:
with blr:
    ppc = pm.sample_posterior_predictive(trace, random_seed=RANDOM_SEED)

# Compare simulated vs observed mortality rate
sim_rates = (
    ppc.posterior_predictive["obs"].values.reshape(-1, len(y_train)).mean(axis=1)
)
obs_rate = y_train.mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    sim_rates,
    bins=40,
    color="steelblue",
    edgecolor="white",
    density=True,
    alpha=0.8,
    label="Simulated rates",
)
ax.axvline(obs_rate, color="red", linewidth=2, label=f"Observed ({obs_rate:.1%})")
ax.set_title(
    "Posterior Predictive Check: Simulated vs Observed Mortality Rate",
    fontweight="bold",
)
ax.set_xlabel("Predicted Mortality Rate")
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 8: Model Evaluation & Benchmark Comparison

We evaluate the model on the held-out test set (20% of data) using **ROC-AUC**,
and compare against the legacy SUPPORT model benchmark (`surv6m`).

**Baselines to beat (from EDA Section):**
- SUPPORT model (`surv6m`): AUC ≈ 0.789
- Physician estimate (`prg6m`): AUC ≈ 0.756
- APACHE III (`aps`): AUC ≈ 0.658

In [ ]:
# Compute posterior predictive probabilities on the test set
post_intercept = trace.posterior["intercept"].values.flatten()
post_betas_all = trace.posterior["betas"].values.reshape(-1, n_features)

# logit_p for each test patient: shape (n_posterior_samples, n_test)
logits_test = post_intercept[:, None] + post_betas_all @ X_test.T
probs_test = sigmoid(logits_test)

pred_mean = probs_test.mean(axis=0)
pred_lower = np.percentile(probs_test, 3, axis=0)
pred_upper = np.percentile(probs_test, 97, axis=0)

# Standard comparison vs the legacy benchmarks (shared helpers in utils.evaluation)
benchmark_preds = get_benchmark_preds(df, idx_test)  # reused by the ROC + Section 9
auc_bayes = roc_auc_score(y_test, pred_mean)
print(
    auc_table(y_test, {"Bayesian LR (linear)": pred_mean}, benchmark_preds).to_string(
        index=False
    )
)

In [ ]:
plot_roc(
    y_test,
    {"Bayesian LR (linear)": pred_mean, **benchmark_preds},
    title="ROC Curves: Bayesian LR vs Legacy Benchmarks",
)
plt.tight_layout()
plt.show()

---
## Section 9: Nonlinear Modeling:  B-splines

The main BLR treats every continuous feature **linearly**. But several day-3 vitals are clinically *U-shaped* in mortality (both extremes are dangerous), and the liver marker `bili` rises non-linearly.

Here we refit the model with **cubic B-splines** on the most non-linear features, keep everything else linear, and check whether it improves held-out AUC / calibration over the linear baseline.

### 9.1 Which features are non-linear?

**(Should put in EDA section instead)** 

For each candidate we split patients into **low** (≤10th pct), **mid** (45–55th pct), and **high** (≥90th pct) by the feature value and compare the observed 180-day mortality in each group:

- **U-shaped**: both low and high mortality exceed the middle (extremes are deadlier than the center).
- **Monotonic**: mortality rises (or falls) steadily from low → high.

We spline the **4 most non-linear** features and leave the rest linear:

- `sod`, `temp`, `resp` — symmetric **U-shapes** (their linear coefficients are ≈ 0 because the two arms cancel).
- `bili` — **monotonic, flat-then-steep** (also heavily right-skewed per EDA); included to test non-linearity beyond U-shapes.

Honorable mentions left linear: `meanbp` and `hrt` are real U's but **one-arm-dominated**, so their linear terms already capture most of the signal. `bun` is essentially flat and `alb` is cleanly monotonic-decreasing.

In [ ]:
# Model-free non-linearity screen: observed mortality at low / mid / high deciles
CANDIDATES = ["sod", "temp", "resp", "bili", "meanbp", "hrt", "bun", "alb"]
rows = []
for f in CANDIDATES:
    x = df_model[f].values
    q = np.quantile(x, [0.10, 0.45, 0.55, 0.90])
    lo = y[x <= q[0]].mean()
    mid = y[(x >= q[1]) & (x <= q[2])].mean()
    hi = y[x >= q[3]].mean()
    if lo > mid and hi > mid:
        shape = "U-shape"
    elif (lo < mid < hi) or (lo > mid > hi):
        shape = "monotonic"
    else:
        shape = "mixed/flat"
    rows.append([f, lo, mid, hi, lo - mid, hi - mid, shape])

nonlin = pd.DataFrame(
    rows, columns=["feature", "lo", "mid", "hi", "lo-mid", "hi-mid", "shape"]
)
print(nonlin.to_string(index=False))

In [ ]:
# 3 U-shaped + bili (monotonic-nonlinear)
SPLINE_FEATURES = ["sod", "temp", "resp", "bili"]

spline_idx = [FEATURE_NAMES.index(f) for f in SPLINE_FEATURES]
keep = [i for i in range(n_features) if i not in spline_idx]
linear_names = [FEATURE_NAMES[i] for i in keep]
X_lin_train = X_train[:, keep]
X_lin_test = X_test[:, keep]

spline_train, spline_test, spline_info = {}, {}, {}
for f in SPLINE_FEATURES:
    xtr = df_model[f].values[idx_train]
    xte = df_model[f].values[idx_test]
    lo_b = float(df_model[f].min())
    hi_b = float(df_model[f].max())
    dm = dmatrix(
        "bs(x, df=5, degree=3, include_intercept=False, "
        "lower_bound=lo_b, upper_bound=hi_b)",
        {"x": xtr, "lo_b": lo_b, "hi_b": hi_b},
        return_type="matrix",
    )
    spline_train[f] = np.asarray(dm)
    spline_info[f] = dm.design_info
    spline_test[f] = np.asarray(build_design_matrices([dm.design_info], {"x": xte})[0])

print(f"Linear features ({len(linear_names)}): {linear_names}")
print(f"Spline features: {SPLINE_FEATURES}")
print(
    "Basis columns per spline feature: "
    f"{[spline_train[f].shape[1] for f in SPLINE_FEATURES]}"
)

In [ ]:
with pm.Model() as blr_spline:
    intercept = pm.Normal("intercept", mu=0, sigma=2)
    betas_lin = pm.Normal("betas_lin", mu=0, sigma=1, shape=X_lin_train.shape[1])
    eta = intercept + pm.math.dot(X_lin_train, betas_lin)

    # One coefficient vector per spline feature; Normal(0, 1) acts as a mild
    # smoothness prior (a random-walk prior would enforce more smoothness if needed).
    for f in SPLINE_FEATURES:
        coef = pm.Normal(f"spline_{f}", mu=0, sigma=1, shape=spline_train[f].shape[1])
        eta = eta + pm.math.dot(spline_train[f], coef)

    p = pm.Deterministic("p", pm.math.invlogit(eta))
    pm.Bernoulli("obs", p=p, observed=y_train)

    trace_spline = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

# Convergence sanity check (same thresholds as Section 5)
sp_vars = ["intercept", "betas_lin"] + [f"spline_{f}" for f in SPLINE_FEATURES]
sp_summary = az.summary(trace_spline, var_names=sp_vars)
print(
    f"Max r_hat: {float(sp_summary['r_hat'].max()):.3f}  |  "
    f"Min ess_bulk: {float(sp_summary['ess_bulk'].min()):.0f}  |  "
    f"Divergences: {int(trace_spline.sample_stats.diverging.values.sum())}"
)

In [ ]:
pm.model_to_graphviz(blr)

In [ ]:
# Reconstruct held-out test probabilities from the spline-model posterior
# (same manual approach as Section 8).
sp_intercept = trace_spline.posterior["intercept"].values.flatten()
sp_betas = trace_spline.posterior["betas_lin"].values.reshape(-1, X_lin_test.shape[1])

logits_sp = sp_intercept[:, None] + sp_betas @ X_lin_test.T
for f in SPLINE_FEATURES:
    c = trace_spline.posterior[f"spline_{f}"].values.reshape(
        -1, spline_test[f].shape[1]
    )
    logits_sp = logits_sp + c @ spline_test[f].T
probs_sp_test = sigmoid(logits_sp)
pred_sp_mean = probs_sp_test.mean(axis=0)

# Standard comparison via the shared helpers (linear baseline + spline + benchmarks)
auc_spline = roc_auc_score(y_test, pred_sp_mean)
model_preds = {
    "Bayesian LR (linear)": pred_mean,
    "Bayesian LR (spline)": pred_sp_mean,
}
print(auc_table(y_test, model_preds, benchmark_preds).to_string(index=False))
print(
    f"\nBrier  linear={brier_score(y_test, pred_mean):.3f}  "
    f"spline={brier_score(y_test, pred_sp_mean):.3f}"
)

plot_roc(
    y_test,
    {**model_preds, **benchmark_preds},
    title="ROC: Linear vs B-spline vs Benchmarks",
)
plt.tight_layout()
plt.show()

In [ ]:
# Partial-effect plots: the centered log-odds contribution of each spline feature,
# with its 94% credible band. This is where the U / non-linear shapes become visible.
fig, axes = plt.subplots(1, len(SPLINE_FEATURES), figsize=(4 * len(SPLINE_FEATURES), 4))
for ax, f in zip(axes, SPLINE_FEATURES):
    xtr = df_model[f].values[idx_train]
    grid = np.linspace(np.percentile(xtr, 1), np.percentile(xtr, 99), 100)
    Bg = np.asarray(build_design_matrices([spline_info[f]], {"x": grid})[0])
    c = trace_spline.posterior[f"spline_{f}"].values.reshape(-1, Bg.shape[1])

    contrib = Bg @ c.T  # (grid, draws)
    contrib = contrib - contrib.mean(axis=0, keepdims=True)  # center for readability
    mean = contrib.mean(axis=1)
    lo = np.percentile(contrib, 3, axis=1)
    hi = np.percentile(contrib, 97, axis=1)

    ax.fill_between(grid, lo, hi, color="steelblue", alpha=0.25)
    ax.plot(grid, mean, color="steelblue", linewidth=2)
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    # rug of the training values to show where the data (and the curve) is trustworthy
    ax.plot(xtr, np.full_like(xtr, lo.min()), "|", color="grey", alpha=0.05)
    ax.set_title(f, fontweight="bold")
    ax.set_xlabel(f"{f} (raw)")
axes[0].set_ylabel("Centered log-odds contribution")
fig.suptitle(
    "B-spline partial effects (U-shapes for sod/temp/resp; flat-then-steep for bili)",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

### 9.2 Interpretation

**Spline Shapes:** the partial-effect curves match the 9.1 screen, `sod`, `temp`, and `resp` bend into the expected **U** (risk lowest near the physiological mid-range and rising at both extremes), while `bili` is **flat at low values then climbs**, the monotonic-nonlinear shape a single linear coefficient could never express.

**Gain is modest:**  AUC moved **0.750 → 0.753 (+0.003)** and Brier **0.201 → 0.200 (−0.001)**. This is expected as these four features have small marginal effects, so even modeling their shape correctly nudges overall discrimination only slightly. 

**Takeaway:** Nonlinear modeling is inefficient for the current dataset and not worth the increase cost and complexity.

**Scope caveat:** `meanbp`/`hrt` were deliberately left linear (one-arm-dominated U's whose linear terms already capture most of the signal).